# Notebook 3: Loss Function and Scalar Backpropagation
## Neural Network from Scratch: Mathematical Derivation and Explicit-Loop Implementation
In the previous notebook, we constructed the forward pass of a one-hidden-layer neural network using scalar calculations and explicit Python loops.
The network maps an input vector
$
\mathbf{x}\in\mathbb{R}^{d}
$
through a hidden layer containing (h) neurons and produces a scalar probability for binary classification.
In this notebook, we derive and implement the corresponding backward pass.

The central question is:
Given the prediction error produced by the network, how does each individual weight and bias contribute to that error?

We will answer this question by applying the chain rule one derivative at a time.

### Objectives
By the end of this notebook, we will:
1. Define binary cross-entropy loss.
2. Derive the output-layer error term.
3. Backpropagate the error into the hidden layer.
4. Derive every weight and bias gradient.
5. Implement backpropagation using explicit loops.
6. Manually verify the gradients on a small synthetic network.
7. Perform numerical gradient checking.
8. Apply one complete forward and backward pass to an MNIST image.

This notebook deliberately avoids vectorized backpropagation. Vectorization will be introduced only after the scalar calculations have been verified.


## 1. Network Architecture and Notation
We use a one-hidden-layer binary classification network.
For an input vector with (d) features and a hidden layer with (h) neurons, the parameter dimensions are:
$$
W^{(1)}\in\mathbb{R}^{h\times d}
$$
$$
\mathbf{b}^{(1)}\in\mathbb{R}^{h}
$$
$$
\mathbf{w}^{(2)}\in\mathbb{R}^{h}
$$
$$
b^{(2)}\in\mathbb{R}
$$
The first-layer weight $W^{(1)}_{ji}$ connects input feature $x_i$ to hidden neuron (j).
The second-layer weight $w^{(2)}_j$ connects hidden neuron (j) to the output neuron.
For each hidden neuron (j):
$$
\sum_{i=1}^{d}W_{ji}^{(1)}x_i+b_j^{(1)}
$$
$$
\sigma\left(z_j^{(1)}\right)
$$
The output pre-activation is:
$$
\sum_{j=1}^{h}w_j^{(2)}a_j^{(1)}+b^{(2)}
$$
The predicted probability is:
$$
a^{(2)}
=\sigma\left(z^{(2)}\right)
$$
where the sigmoid function is:
$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$
The model predicts the probability that the input belongs to class (1).

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=8, suppress=True)

## 2. Scalar Sigmoid Function and Its Derivative
The sigmoid activation function is:
$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$
Its derivative is:
$$
\sigma(z)\left(1-\sigma(z)\right)
$$
Because the activation value $a=\sigma(z)$ is already calculated during forward propagation, we can write the derivative as:
$$
\sigma'(z)=a(1-a)
$$
This form avoids recalculating the exponential function during backpropagation.

In [6]:
def sigmoid_scalar(z):
    if z >= 0:
        return 1.0 / (1.0 + math.exp(-z))

    exp_z = math.exp(z)
    return exp_z / (1.0 + exp_z)
    
def sigmoid_derivative_from_activation(a):
    return a * (1.0 - a)

In [7]:
test_values = [-3.0, 0.0, 3.0]
for z in test_values:
    activation = sigmoid_scalar(z)
    derivative = sigmoid_derivative_from_activation(activation)
    print(
        f"z = {z:5.1f}, "
        f"sigmoid(z) = {activation:.6f}, "
        f"sigmoid'(z) = {derivative:.6f}"
    )

z =  -3.0, sigmoid(z) = 0.047426, sigmoid'(z) = 0.045177
z =   0.0, sigmoid(z) = 0.500000, sigmoid'(z) = 0.250000
z =   3.0, sigmoid(z) = 0.952574, sigmoid'(z) = 0.045177


### Analysis
The sigmoid derivative is largest when $z=0$, where:
$$
\sigma(0)=0.5
$$
and:
$$
\sigma'(0)=0.5(1-0.5)=0.25
$$
The derivative becomes small when the activation approaches either (0) or (1). This behavior will later help explain the vanishing-gradient problem in deeper sigmoid networks.


## 3. Binary Cross-Entropy Loss
For binary classification, the true label satisfies:
$$
y\in{0,1}
$$
and the network produces:
$$
\hat{y}\in(0,1)
$$
The binary cross-entropy loss for one observation is:
$$
-\left[
y\log(\hat{y})
+
(1-y)\log(1-\hat{y})
\right]
$$
The formula contains two cases.

Case 1: $y=1$
$$
\mathcal{L}(1,\hat{y})=-\log(\hat{y})
$$
The loss becomes small when $\hat{y}$ is close to (1).

Case 2: $y=0$
$$
\mathcal{L}(0,\hat{y})=-\log(1-\hat{y})
$$
The loss becomes small when $\hat{y}$ is close to (0).
Binary cross-entropy therefore penalizes confident incorrect predictions much more strongly than uncertain predictions.

In [9]:
def binary_cross_entropy_scalar(y, y_hat, epsilon=1e-12):
    y_hat_clipped = min(max(y_hat, epsilon), 1.0 - epsilon)
    bce = -(y * math.log(y_hat_clipped) + (1.0 - y) * math.log(1.0 - y_hat_clipped))
    
    return bce

In [11]:
examples = [
(1.0, 0.99),
(1.0, 0.70),
(1.0, 0.10),
(0.0, 0.01),
(0.0, 0.30),
(0.0, 0.90),
]

for y, y_hat in examples:
    loss = binary_cross_entropy_scalar(y, y_hat)
    print(
        f"y = {y:.0f}, "
        f"y_hat = {y_hat:.2f}, "
        f"loss = {loss:.6f}"
    )

y = 1, y_hat = 0.99, loss = 0.010050
y = 1, y_hat = 0.70, loss = 0.356675
y = 1, y_hat = 0.10, loss = 2.302585
y = 0, y_hat = 0.01, loss = 0.010050
y = 0, y_hat = 0.30, loss = 0.356675
y = 0, y_hat = 0.90, loss = 2.302585


### Analysis
When the true class is (1), increasing $\hat{y}$ decreases the loss.
When the true class is (0), decreasing $\hat{y}$ decreases the loss.
A confidently incorrect prediction such as:
$$
y=1,\qquad \hat{y}=0.10
$$
produces a much larger loss than a less confident prediction such as:
$$
y=1,\qquad \hat{y}=0.70
$$
This makes binary cross-entropy especially appropriate for probability-based binary classification.

## 4. Deriving the Output-Layer Gradient
The output neuron performs two operations:
$$
\sum_{j=1}^{h}w_j^{(2)}a_j^{(1)}+b^{(2)}
$$
$$
\hat{y}=\sigma\left(z^{(2)}\right)
$$
The loss is:
$$
-\left[
y\log(\hat{y})
+
(1-y)\log(1-\hat{y})
\right]
$$
To update the network, we first need:
$$
\frac{\partial\mathcal{L}}{\partial z^{(2)}}
$$
Using the chain rule:
$$
\frac{\partial\mathcal{L}}{\partial\hat{y}}
\frac{\partial\hat{y}}{\partial z^{(2)}}
$$
First, differentiate the loss with respect to $\hat{y}$:
$$
-\frac{y}{\hat{y}}
+
\frac{1-y}{1-\hat{y}}
$$
The derivative of the sigmoid output is:
$$
\hat{y}(1-\hat{y})
$$
Multiplying the two expressions gives:
$$
\left(
-\frac{y}{\hat{y}}
+
\frac{1-y}{1-\hat{y}}
\right)
\hat{y}(1-\hat{y})
$$
After simplification:
$$
\hat{y}-y
$$
We define the output-layer error term as:
$$
\boxed{
\delta^{(2)}=\hat{y}-y
}
$$
This simplification occurs specifically because binary cross-entropy is combined with a sigmoid output.

## 5. Output-Layer Weight and Bias Gradients
The output pre-activation is:
$$
\sum_{j=1}^{h}w_j^{(2)}a_j^{(1)}+b^{(2)}
$$
For output weight $w_j^{(2)}$:
$$
a_j^{(1)}
$$
Using the chain rule:
$$
\frac{\partial\mathcal{L}}{\partial z^{(2)}}
\frac{\partial z^{(2)}}{\partial w_j^{(2)}}
$$
Therefore:
$$
\delta^{(2)}a_j^{(1)}
$$

For the output bias:
$$
\frac{\partial z^{(2)}}{\partial b^{(2)}}=1
$$
Therefore:
$$
\delta^{(2)}
$$

The output-layer gradients are therefore:
$$
\frac{\partial\mathcal{L}}{\partial w_j^{(2)}}=
(\hat{y}-y)a_j^{(1)}
$$
$$
\frac{\partial\mathcal{L}}{\partial b^{(2)}}=
\hat{y}-y
$$

## 6. Backpropagating into the Hidden Layer
Hidden neuron (j) influences the loss through the following path:
$$
z_j^{(1)}
\rightarrow
a_j^{(1)}
\rightarrow
z^{(2)}
\rightarrow
\hat{y}
\rightarrow
\mathcal{L}
$$
We need:
$$
\frac{\partial\mathcal{L}}{\partial z_j^{(1)}}
$$
Applying the chain rule:
$$
\frac{\partial\mathcal{L}}{\partial z^{(2)}}
\frac{\partial z^{(2)}}{\partial a_j^{(1)}}
\frac{\partial a_j^{(1)}}{\partial z_j^{(1)}}
$$
We already know:
$$
\frac{\partial\mathcal{L}}{\partial z^{(2)}}=
\delta^{(2)}
$$
Because:
$$
z^{(2)}=
\sum_{j=1}^{h}w_j^{(2)}a_j^{(1)}+b^{(2)}
$$
we have:
$$
\frac{\partial z^{(2)}}{\partial a_j^{(1)}}=w_j^{(2)}
$$
The hidden activation is sigmoid, so:
$$
\frac{\partial a_j^{(1)}}{\partial z_j^{(1)}}=
a_j^{(1)}\left(1-a_j^{(1)}\right)
$$
Combining these terms:
$$
\delta^{(2)}
w_j^{(2)}
a_j^{(1)}
\left(1-a_j^{(1)}\right)
$$
We define the hidden-layer error term:
$$
\delta^{(1)}=
\delta^{(2)}
w_j^{(2)}
a_j^{(1)}
\left(1-a_j^{(1)}\right)
$$
This term measures how much hidden neuron (j)'s pre-activation contributed to the final loss.

## 7. Hidden-Layer Weight and Bias Gradients
The pre-activation of hidden neuron (j) is:
$$
\sum_{i=1}^{d}W_{ji}^{(1)}x_i+b_j^{(1)}
$$
For the weight connecting input (i) to hidden neuron (j):
$$
x_i
$$
Therefore:
$$
\frac{\partial\mathcal{L}}{\partial z_j^{(1)}}
\frac{\partial z_j^{(1)}}{\partial W_{ji}^{(1)}}
$$
which gives:
$$
\delta_j^{(1)}x_i
$$
For the hidden bias:
$$
\frac{\partial z_j^{(1)}}{\partial b_j^{(1)}}=1
$$
Therefore:
$$
\delta_j^{(1)}
$$

The full scalar backpropagation equations are:
$$
\delta^{(2)}=\hat{y}-y
$$
$$
\frac{\partial\mathcal{L}}{\partial w_j^{(2)}}=
\delta^{(2)}a_j^{(1)}
$$
$$
\frac{\partial\mathcal{L}}{\partial b^{(2)}}=
\delta^{(2)}
$$
$$
\delta^{(1)}=
\delta^{(2)}
w_j^{(2)}
a_j^{(1)}
\left(1-a_j^{(1)}\right)
$$
$$
\frac{\partial\mathcal{L}}{\partial W_{ji}^{(1)}}=
\delta_j^{(1)}x_i
$$
$$
\frac{\partial\mathcal{L}}{\partial b_j^{(1)}}=
\delta_j^{(1)}
$$
